# Generating Lorenz-63 Truth Data

This notebook shows how to generate a Lorenz-63 "truth" trajectory -- the synthetic ground-truth state used both as the observed input to, and the validation target for, a data assimilation filter (e.g. the CGKF Lorenz-63 example in `notebooks/CGKF/exploration/lorenz63.ipynb`).

**Setup** (once, from the repo root):
```bash
pip install -e .
```
This notebook assumes it is run with the repo root (`data_assimilation/`) as the working directory.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from da.data import load_lorenz63
from da.models.lorenz63 import EM
from da.result import AssimilationResult

## The model

Lorenz-63 is generated here as a *stochastic* system (an SDE, not the classic deterministic ODE), so that it fits the conditionally-Gaussian framework the filters expect:

$$
\begin{aligned}
dx &= \sigma(y - x)\, dt, \\
dy &= \big(x(\rho - z) - y\big)\, dt + c\, dW_t, \\
dz &= (xy - \beta z)\, dt,
\end{aligned}
$$

integrated with simple Euler-Maruyama. `x` is treated as the observed state (`U`); `(y, z)` are the hidden states (`mu`).

**Note the asymmetry:** only the `y` equation carries a stochastic forcing term (`c * dW_t`) in this implementation -- `x` and `z` are otherwise driven deterministically by `x`, `y`, `z`. `c` sets that noise's amplitude.

Default parameters (hard-coded in `EM`, the standard chaotic regime): `sigma=10`, `rho=28`, `beta=8/3`.

In [ ]:
# Reproducibility: EM() does not seed the RNG itself, so set a seed if you want
# the same trajectory across runs.
np.random.seed(0)

true_obj = load_lorenz63()  # defaults: T=100, timesteps=100000 (dt=1e-3), c=1, x0=[0, 1.0, 0]

print(f"t: {true_obj.t.shape}, spans [{true_obj.t[0]}, {true_obj.t[-1]}]")
print(f"U (observed x): {true_obj.U.shape}")
print(f"mu (hidden y, z): {true_obj.mu.shape}")
print(f"system: {true_obj.system}")

## What `load_lorenz63()` returns

`load_lorenz63()` is a thin wrapper: it calls `EM(x0=[0, 1.0, 0], T=T, timesteps=timesteps, c=c)` and packages the result into an `AssimilationResult` (`t`, `U` = x, `mu` = [y, z], `system="Lorenz63"`). This is exactly the object the CGKF notebook's `CGKF(...)` call and `run_l63_dashboard(...)` expect as `true_obj` -- generate it here, pass it there.

**Limitation to be aware of:** `load_lorenz63()` only exposes `T`, `timesteps`, and `c` -- the initial condition `x0=[0, 1.0, 0]` is fixed inside the function and is *not* a parameter you can override through this entry point. If you need a different initial condition, call `EM()` directly (see below).

**Transient:** the trajectory starts off the attractor at `[0, 1.0, 0]`, so the first ~1-2 time units are a relaxation transient, not representative chaotic-attractor statistics. This matches how truth is generated elsewhere in this repo (e.g. `scripts/run_l63_comparison.py`), so it's consistent, but worth knowing if you're computing statistics that are sensitive to it.

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(true_obj.U, true_obj.mu[:, 0], true_obj.mu[:, 1],
                 c=true_obj.t, cmap="viridis", s=0.3, linewidths=0)
ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_zlabel("$z$")
ax.set_title("Lorenz-63 Attractor (Generated Truth)")
fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.1, label="time")
plt.tight_layout()
plt.savefig("scratch/output/lorenz63_generation/attractor.png", dpi=200)
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].plot(true_obj.t, true_obj.U, lw=0.5); axes[0].set_ylabel("$x$ (observed)")
axes[1].plot(true_obj.t, true_obj.mu[:, 0], lw=0.5, color="tab:orange"); axes[1].set_ylabel("$y$ (hidden)")
axes[2].plot(true_obj.t, true_obj.mu[:, 1], lw=0.5, color="tab:green"); axes[2].set_ylabel("$z$ (hidden)")
axes[2].set_xlabel("time")
axes[0].set_title("Lorenz-63 Time Series (Generated Truth)")
plt.tight_layout()
plt.savefig("scratch/output/lorenz63_generation/timeseries.png", dpi=200)
plt.show()

## Customizing generation: calling `EM()` directly

For anything `load_lorenz63()` doesn't expose -- a different initial condition, a longer/shorter run, a different noise level -- call the underlying integrator, `EM()`, yourself and wrap the result in an `AssimilationResult`:

- `x0`: initial condition, `[x0, y0, z0]`
- `T`: total time span
- `timesteps`: number of steps (sets `dt = T / timesteps`)
- `c`: stochastic forcing amplitude on the `y` equation (`EM`'s own default is `c=10`; `load_lorenz63` uses a gentler `c=1`)

In [ ]:
np.random.seed(0)

data = EM(x0=np.array([1.0, 1.0, 1.0]), T=50, timesteps=50_000, c=2)
t_custom = np.linspace(0, 50, 50_000)

custom_obj = AssimilationResult(
    t_custom,
    data[:, 0],       # x -> observed
    data[:, 1:3],     # (y, z) -> hidden
    system="Lorenz63",
)

print(f"t: {custom_obj.t.shape}, U: {custom_obj.U.shape}, mu: {custom_obj.mu.shape}")

## Saving a generated trajectory for reuse

Lorenz-63 is cheap to regenerate on the fly (a fraction of a second for 100k steps), so saving to disk usually isn't necessary -- just re-run `load_lorenz63()` with the same seed. If you do want a fixed trajectory to share (e.g. to guarantee collaborators use the exact same truth), save the seed alongside the arrays:

In [ ]:
np.savez(
    "scratch/output/lorenz63_generation/lorenz63_truth.npz",
    t=true_obj.t, U=true_obj.U, mu=true_obj.mu, seed=0,
)
print("Saved to scratch/output/lorenz63_generation/lorenz63_truth.npz")
# To reload: AssimilationResult(d["t"], d["U"], d["mu"], system="Lorenz63") where d = np.load(path)

## Next step

With `true_obj` in hand, head to `notebooks/CGKF/exploration/lorenz63.ipynb` to run the CGKF filter against it.